In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

test_connection = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=ChurnDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
print("connection successfully")
 
# Connect Python to your SQL Server & Python Data Cleaning & Feature Preparation
connection_string = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=ChurnDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)


query = "SELECT * FROM dbo.vw_ChurnData"

df = pd.read_sql(query, engine)

print(df.shape)
print(df.info())
print(df.head())

connection successfully
(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  Paperle

In [2]:
# Python Data Cleaning & Feature Preparation

# check missing or blank values
data=df.isnull().sum()
print(data)

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [3]:
# Handle the 11 missing TotalCharges
df["TotalCharges"]=df["TotalCharges"].fillna(0)
print(df["TotalCharges"].isnull().sum())

0


In [4]:
# Convert Yes/No columns into 1/0.
binary_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})

print(df[binary_columns].head())

   Partner  Dependents  PhoneService  PaperlessBilling  Churn
0        1           0             0                 1      0
1        0           0             1                 0      0
2        0           0             1                 1      1
3        0           0             0                 0      0
4        0           0             1                 1      1


In [5]:
# Remove CustomerID
df=df.drop('customerID',axis=1)
print(df.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [6]:
df1=df.copy()

In [7]:
# One-Hot Encoding for the cleaned dataset
categorical_columns = [
    'gender',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod',
    
]
df=pd.get_dummies(df,columns=categorical_columns,drop_first=True,dtype=int)

print(df.dtypes)
print(df.head())

SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
gender_Male                                int64
MultipleLines_No phone service             int64
MultipleLines_Yes                          int64
InternetService_Fiber optic                int64
InternetService_No                         int64
OnlineSecurity_No internet service         int64
OnlineSecurity_Yes                         int64
OnlineBackup_No internet service           int64
OnlineBackup_Yes                           int64
DeviceProtection_No internet service       int64
DeviceProtection_Yes                       int64
TechSupport_No inter

In [8]:
# Save the cleaned dataset
df1.to_csv("cleaned_churn_data.csv", index=False)
df.to_csv(r"C:\Users\Surya\OneDrive\Desktop\CUSTOMER CHURN PROJECT\cleaned_churn_data.csv",index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [9]:
# Feature Engineering for Machine Learning

# Feature 1 — TotalServicesUsed
services_columns=[
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]
df1['TotalServicesUsed'] = df1[services_columns].apply(lambda row: (row == 'Yes').sum(),axis=1)
   
print(df1[['TotalServicesUsed']])    

      TotalServicesUsed
0                     1
1                     2
2                     2
3                     3
4                     0
...                 ...
7038                  5
7039                  4
7040                  1
7041                  0
7042                  5

[7043 rows x 1 columns]


In [10]:
# Feature 2 — CustomerSegment
df1["Customer_segment"]=pd.cut(
    df1["tenure"],
    bins=[-1,12,24,float('inf')],
    labels=["New","Established","Loyal"])
print(df1[["tenure","Customer_segment"]])

      tenure Customer_segment
0          1              New
1         34            Loyal
2          2              New
3         45            Loyal
4          2              New
...      ...              ...
7038      24      Established
7039      72            Loyal
7040      11              New
7041       4              New
7042      66            Loyal

[7043 rows x 2 columns]


In [11]:
# check correlation for TotalServicesUsed with the churn target

df1["TotalServicesUsed"].corr(df1["Churn"])
print(df1["TotalServicesUsed"])

0       1
1       2
2       2
3       3
4       0
       ..
7038    5
7039    4
7040    1
7041    0
7042    5
Name: TotalServicesUsed, Length: 7043, dtype: int64


In [12]:
# check correlation for Customer_segment with the churn target

df1.groupby("Customer_segment")["Churn"].mean()*100

C:\Users\Surya\AppData\Local\Temp\ipykernel_35028\3350317944.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df1.groupby("Customer_segment")["Churn"].mean()*100


Customer_segment
New            47.438243
Established    28.710938
Loyal          14.036003
Name: Churn, dtype: float64

In [13]:
# TotalServicesUsed

 # I created TotalServicesUsed to show how many additional services each customer is using.
 # I kept this feature because customers using different numbers of services may have different churn patterns.

# CustomerSegment

 # I created CustomerSegment based on how long the customer has been with the company, grouping them as New, Established, or Loyal. 
 # I kept this feature because customers at different stages of their relationship with the company may have different churn behavior.

In [14]:
# One-Hot Encoding with Customer_segment for the cleaned & featured dataset
categorical_columns = [
    'gender',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod',
    'Customer_segment'
    
]
df1=pd.get_dummies(df1,columns=categorical_columns,drop_first=True,dtype=int)


In [15]:
# check blank values and datatypes
print(df1.isnull().sum().sum())
print("Final shape:", df1.shape)
print(df1.dtypes)

0
Final shape: (7043, 34)
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
TotalServicesUsed                          int64
gender_Male                                int64
MultipleLines_No phone service             int64
MultipleLines_Yes                          int64
InternetService_Fiber optic                int64
InternetService_No                         int64
OnlineSecurity_No internet service         int64
OnlineSecurity_Yes                         int64
OnlineBackup_No internet service           int64
OnlineBackup_Yes                           int64
DeviceProtection_No internet service       

In [16]:
print(df1.head())

   SeniorCitizen  Partner  Dependents  tenure  PhoneService  PaperlessBilling  \
0              0        1           0       1             0                 1   
1              0        0           0      34             1                 0   
2              0        0           0       2             1                 1   
3              0        0           0      45             0                 0   
4              0        0           0       2             1                 1   

   MonthlyCharges  TotalCharges  Churn  TotalServicesUsed  ...  \
0           29.85         29.85      0                  1  ...   
1           56.95       1889.50      0                  2  ...   
2           53.85        108.15      1                  2  ...   
3           42.30       1840.75      0                  3  ...   
4           70.70        151.65      1                  0  ...   

   StreamingTV_Yes  StreamingMovies_No internet service  StreamingMovies_Yes  \
0                0                  

In [17]:
# saved the cleaned_featured_churn_dataset
df1.to_csv("cleaned_featured_churn_data.csv", index=False)
df1.to_csv(r"C:\Users\Surya\OneDrive\Desktop\CUSTOMER CHURN PROJECT\cleaned_featured_churn_data.csv",index=False)

print("Cleaned featured dataset saved successfully!")

Cleaned featured dataset saved successfully!
